# Этап 2: обучение LoRA

Среда выполнения: GPU (T4). Результаты пишутся на Google Drive: адаптер, контрольные точки, ответы модели.
Оборвалась сессия - запустить ячейки заново: обучение продолжится с последней контрольной точки.
Ответы (`*.jsonl`) скачать с Drive и оценить на ноутбуке: `python src/mcnl/evaluate.py --split dev --data data/dataset/v2 --predictions ответы.jsonl`.

In [ ]:
!pip -q install -U transformers peft accelerate bitsandbytes
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!rm -rf /content/repo && git clone -q --depth 1 https://github.com/xdstk/lang.cir.spice.mc12 /content/repo
%cd /content/repo
!git log --oneline -1

In [ ]:
# настройки запуска
MODEL = 'ЗАПОЛНИТЬ: id модели на Hugging Face'
RUN = 'smoke1'         # имя запуска: папка на Drive
SUBSET = 2000          # 1000, 2000 или 3000 записей train v2
FORMAT = 'netlist'     # netlist или ir
EPOCHS = 3
SEED = 1
OUT = f'/content/drive/MyDrive/mcnl/{RUN}'

In [ ]:
!python src/mcnl/training.py --data data/dataset/v2 --subset {SUBSET} --format {FORMAT} --out out/training/{RUN}

Модель без обучения: нужна один раз на модель, можно пропустить.

In [ ]:
!mkdir -p {OUT}
!python colab/predict.py --model {MODEL} --format {FORMAT} --records data/dataset/v2/dev.jsonl --out {OUT}/untrained_dev.jsonl --name untrained

In [ ]:
!python colab/train_lora.py --model {MODEL} --data out/training/{RUN} --out {OUT} --epochs {EPOCHS} --seed {SEED}

In [ ]:
!python colab/predict.py --model {MODEL} --adapter {OUT}/adapter --format {FORMAT} --records data/dataset/v2/dev.jsonl --out {OUT}/dev.jsonl --name {RUN}
!python colab/predict.py --model {MODEL} --adapter {OUT}/adapter --format {FORMAT} --records data/human_test/v1/paraphrase.jsonl --out {OUT}/paraphrase.jsonl --name {RUN}
!ls -la {OUT}